# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [5]:
df.shape

(100000, 11)

In [ ]:
df["quality rating"].value_counts()

quality rating
10    10143
5     10124
1     10090
2     10030
8      9977
6      9961
9      9955
7      9954
4      9928
3      9838
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [13]:
# Geçenler True(1), Kalanlar False(0) oluyor
y = (df["quality rating"] >= 6).astype(int)

# İlk 5 numunenin yeni QC etiketlerini kontrol edelim
print(y.head())

0    1
1    1
2    0
3    1
4    0
Name: quality rating, dtype: int64


❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [ ]:
y.value_counts()

quality rating
0    50010
1    49990
Name: count, dtype: int64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [16]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=["quality rating"]))

print(X_scaled[:5])

[[-0.78860255 -1.52846058 -1.7331803   0.4611296  -1.52665289 -0.85238094
  -0.22139287 -0.47838741  0.34023114 -0.48983293]
 [-0.34686003 -0.46206949 -0.15828964 -0.78386767  0.11706606 -3.1026336
  -0.30135681 -0.98697188 -0.76942881 -0.48983293]
 [ 0.06441748  0.23275676  0.55041116  1.01755296 -0.43911679  0.42343195
   1.4578499   1.24681087  1.33992478 -0.30738714]
 [ 0.37668374 -0.61069543 -1.10322403  0.09945442 -1.01516331  0.92572049
  -3.03012631  1.06731047  0.6601331  -0.42667862]
 [ 1.22970378  1.24712878  0.22887099  0.30811318  0.58386238 -0.62132821
   0.21840881 -0.95705514 -2.31895396  1.32059075]]


## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [ ]:
import time
from sklearn.linear_model import LogisticRegression

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

for solver in solvers:
    # 1. Kronometreyi başlat
    start_time = time.time()

    # 2. Modeli tanımla
    model = LogisticRegression(solver=solver)

    # 3. Reaktörü çalıştır (Hem X_scaled hem de y'yi vererek modeli eğit)
    model.fit(X_scaled, y)

    # 4. Bitiş zamanını kaydet
    stop_time = time.time()

    # 5. Süreyi hesapla
    gecen_sure = stop_time - start_time

    # 6. Modelin doğruluğunu (accuracy) ölç
    skor = model.score(X_scaled, y)

    # Sonuçları ekrana yazdır (Burası hazır)
    print(f"Çözücü: {solver:<10} | Süre: {gecen_sure:.4f} saniye | Doğruluk: {skor:.4f}")

Çözücü: newton-cg  | Süre: 0.1280 saniye | Doğruluk: 0.8611
Çözücü: lbfgs      | Süre: 0.0341 saniye | Doğruluk: 0.8611
Çözücü: liblinear  | Süre: 0.0994 saniye | Doğruluk: 0.8612
Çözücü: sag        | Süre: 0.4735 saniye | Doğruluk: 0.8612
Çözücü: saga       | Süre: 0.7706 saniye | Doğruluk: 0.8612


In [18]:
# YOUR ANSWER
fastest_solver = "lbfgs"

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [19]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/didemarslan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/didemarslan/reading_the_green_line/S16D1-S-data-knn/S16D3-S-data-electrocardiograms/S16D3-S-data-threshold/S16D4-S-loss-functions/S16D4-S-data-solvers/tests
plugins: dash-4.4.1, langsmith-0.12.4, typeguard-4.4.2, anyio-4.15.1
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [23]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_validate
import time

start_time = time.time()

# Cihazı 'Lojistik Regresyon' modunda çalıştırmak için loss='log_loss' ekliyoruz
model = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3)

# Kalite kontrol raporlama birimini 'accuracy' olarak değiştiriyoruz
cv_results = cross_validate(model, X_scaled, y, cv=5, scoring='accuracy')

stop_time = time.time()
gecen_sure = stop_time - start_time

print(f"Süre: {gecen_sure:.4f} saniye")
print(f"Ortalama Kesinlik (Accuracy) Skoru: {cv_results['test_score'].mean():.4f}")

Süre: 0.5685 saniye
Ortalama Kesinlik (Accuracy) Skoru: 0.8592


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [24]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [26]:
# Cihazın ana kalibrasyonunu yapıyoruz (Reaktörü tüm ham maddeyle çalıştırıyoruz)
model.fit(X_scaled, y)
# ----------------------------------------

# 1. Ön İşleme: Yeni numuneyi mevcut laboratuvar standartlarına (ölçeğe) getiriyoruz
new_wine_scaled = scaler.transform(new_wine)

# 2. Sınıflandırma: Modelden (cihazdan) kesin kararı (Geçti:1 / Kaldı:0) alıyoruz
predicted_class = model.predict(new_wine_scaled)[0]

# 3. Güven Skoru: Modelin bu kararı verirkenki emin olma oranını (olasılığını) hesaplıyoruz
predicted_proba_of_class = model.predict_proba(new_wine_scaled)[0][1]

print(f"Tahmin Edilen Sınıf: {predicted_class}")
print(f"İyi Şarap (1) Olma Olasılığı: % {predicted_proba_of_class * 100:.2f}")

Tahmin Edilen Sınıf: 0
İyi Şarap (1) Olma Olasılığı: % 2.93


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/didemarslan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/didemarslan/reading_the_green_line/S16D1-S-data-knn/S16D3-S-data-electrocardiograms/S16D3-S-data-threshold/S16D4-S-loss-functions/S16D4-S-data-solvers/tests
plugins: dash-4.4.1, langsmith-0.12.4, typeguard-4.4.2, anyio-4.15.1
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba FAILED [100%]

=================================== FAILURES ===================================
__________________ TestNewDataPrediction.test_predicted_proba __________________

self = <tests.test_new_data_prediction.TestNewDataPrediction testMethod=test_predicted_proba>

    def test_predicted_proba(self):
>       self.ass